In [ ]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))
import itertools
import concurrent.futures
from libs import constants as Cs
import os
import json
import datetime
import pickle
import optuna

SEEDS = list(range(101, 115))
TEST_EVAL_EPS = 5
# lambda fit archving [FrozenTrial(number=80, state=<TrialState.COMPLETE: 1>, values=[118.11550385203829], datetime_start=datetime.datetime(2026, 5, 22, 5, 4, 53, 822616), datetime_complete=datetime.datetime(2026, 5, 22, 5, 36, 8, 359073), params={'crossmethod': 'uniform', 'lambda': 70, 'mu': 70, 'mutation_rate': 0.01, 'cross_rate': 0.9000000000000001, 'sigma': 0.5, 'archiving_period': 4, 'archive_batch': 5, 'cross': 0.6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mu': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.1, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'sigma': FloatDistribution(high=3.0, log=False, low=0.5, step=0.5), 'archiving_period': IntDistribution(high=5, log=False, low=2, step=1), 'archive_batch': IntDistribution(high=5, log=False, low=1, step=1), 'cross': FloatDistribution(high=0.9, log=False, low=0.1, step=0.1)}, trial_id=81, value=None), FrozenTrial(number=86, state=<TrialState.COMPLETE: 1>, values=[118.11550385203829], datetime_start=datetime.datetime(2026, 5, 22, 5, 36, 31, 908497), datetime_complete=datetime.datetime(2026, 5, 22, 6, 11, 5, 128793), params={'crossmethod': 'uniform', 'lambda': 70, 'mu': 70, 'mutation_rate': 0.01, 'cross_rate': 0.9000000000000001, 'sigma': 0.5, 'archiving_period': 4, 'archive_batch': 5, 'cross': 0.6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mu': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.1, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'sigma': FloatDistribution(high=3.0, log=False, low=0.5, step=0.5), 'archiving_period': IntDistribution(high=5, log=False, low=2, step=1), 'archive_batch': IntDistribution(high=5, log=False, low=1, step=1), 'cross': FloatDistribution(high=0.9, log=False, low=0.1, step=0.1)}, trial_id=87, value=None)]
# lambda - this one is bad actually novelty [FrozenTrial(number=4, state=<TrialState.COMPLETE: 1>, values=[-9.291787437438707], datetime_start=datetime.datetime(2026, 5, 19, 22, 25, 13, 886515), datetime_complete=datetime.datetime(2026, 5, 19, 23, 0, 16, 677521), params={'crossmethod': 'mean', 'lambda': 60, 'mu': 60, 'mutation_rate': 0.18, 'cross_rate': 1.0, 'sigma': 2.5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mu': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.5, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'sigma': FloatDistribution(high=3.0, log=False, low=0.5, step=0.5)}, trial_id=208, value=None)]
EXAMPLE_MAPPING  = {
    "lambda":"l",
    "mu": "m",
    "mutation_rate":"mr",
    "cross_rate":"cr",
    "sigma": "mutation_sigma",
    "cross":"cross_uni",
    "crossmethod":"cross_method"
}
GENS = {
    "cartpole": [5, 10, 15, 20, 25],
    "lunarlander":[10, 15, 25, 35, 40, 50, 70]
}
def rename(dict):
    return {EXAMPLE_MAPPING.get(k, k): v for k, v in dict.items()}

def task_job(en, alg, container, args, s, out_path):
    env = Cs.ENIVROMENTS[en]()
    df, pop = Cs.ALG_MAPPING[alg].argumented_function(env=en, container=container, seed=s, out_path=out_path, **args)
    print("Testing " + str(s))
    fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for p in pop]
    print("Finished seed %d of algorithm %s" % (s, alg))
    return fitnesses, pop


def evaluation_of_setup(en, alg, container, out_path, **kwargs):
    #we do not deviate the sigma
    # first we evaluate the current setup

    stat_futures = {}
    gens_future={}
    args = rename(kwargs)
    dirpath = os.path.join(os.path.realpath(out_path), container,alg)

    with concurrent.futures.ProcessPoolExecutor(max_workers=5) as executor:
        print("Launching " + alg + "on Enviroment " + str(en))
        for ng, s in itertools.product(GENS[en], SEEDS):
            args["ng"] = ng
            future = executor.submit(task_job, container=container, alg=alg, en=en, args=args, s=s, out_path=dirpath + "/stat")
            stat_futures[future] = s
            gens_future[future] = ng

    stats = {}
    pops = {}
    for future in concurrent.futures.as_completed(stat_futures):
        s = stat_futures[future]
        ng = gens_future[future]
        result = future.result()
        if "fitness" not in stats:
            stats["fitness"] = {}
        if ng not in stats["fitness"]:
            stats["fitness"][ng] = {}
        stats["fitness"][ng][s] = result[0]
        pops[s]= result[1] 
    ts = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"{ts}_{container}_"+ "|".join(f"{k}: {v}" for k, v in args.items())
    stats["environment"] = en
    stats["algorithm"] = alg
    stats["container"] = container
    stats["arguments"] = args
    json_path = os.path.join(dirpath, filename + ".json")
    pickle_path = os.path.join(dirpath, filename + ".plk")

    with open(json_path, "w") as json_file:
        json.dump(stats,json_file)
        print("Finished "+ filename)
    with open(pickle_path, "wb") as f:
        pickle.dump(pops, f)
    return pops, stats



In [7]:
import numpy as np

def select_minimal_exaples(examples):
    pop = np.inf
    selected_trials = []
    for t in examples:
        k = t["lambda"]+t["mu"]
        if pop == k:
            selected_trials.append(t)
        elif pop > k:
            pop = k
            selected_trials = [t]

    return selected_trials


def make_gen_examination(en, alg, container):
    with open("../relevant_studies.json", "r") as f:
        relevant_study_names = json.load(f)
    storage = f"sqlite:///Data/optuna/{en}/{container}/{alg}.db"
    study_name = relevant_study_names[container][alg][en]

    new_study = optuna.load_study(study_name=study_name,storage=storage)
    most_promising = select_minimal_exaples([t.params for t in new_study.best_trials])
    pops, stats = evaluation_of_setup(
        en=en, 
        alg=alg, 
        container=container,
        out_path=f"./Data/final/{en}",
        **most_promising[0]
    )
    return pops, stats

pops, stats = make_gen_examination(en="lunarlander", alg="lambda", container="fitness")

Launching lambdaon Enviroment lunarlander


2026-06-03 06:59:42.353982: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-06-03 06:59:42.363334: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
2026-06-03 06:59:42.583739: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/li

gen	nevals	avg     	min     	max     	std    
0  	50    	-386.769	-718.135	-106.009	191.229
gen	nevals	avg     	min     	max     	std    
0  	50    	-376.934	-616.413	-125.803	145.317
gen	nevals	avg     	min     	max     	std    
0  	50    	-430.693	-682.896	-67.5599	185.759
gen	nevals	avg     	min     	max    	std    
0  	50    	-442.715	-731.976	-100.56	174.705
gen	nevals	avg     	min     	max    	std    
0  	50    	-500.533	-766.767	-59.416	202.585
1  	38    	-235.012	-490.694	-80.4264	114.461
1  	39    	-246.546	-571.405	-106.009	125.197
1  	40    	-269.781	-580.951	-67.5599	144.756
1  	40    	-265.66 	-572.615	-50.3354	139.155
1  	40    	-311.806	-706.942	-59.416	169.455
2  	42    	-183.685	-371.741	-80.4264	79.5754
2  	39    	-143.831	-475.759	-50.3354	82.4207
2  	43    	-168.46 	-447.087	-43.951 	89.1445
2  	44    	-192.392	-418.607	-67.5599	86.4661
2  	41    	-224.655	-537.586	-59.416	120.647
3  	42    	-126.03 	-231.711	-80.4264	32.5855
3  	41    	-105.918	-208.092	-50.3354	34

/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
2026-06-03 09:01:02.410912: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-06-03 09:01:02

36 	39    	115.724   	115.724 	115.724 	1.42109e-14
35 	38    	97.2136 	97.2136 	97.2136 	1.42109e-14
38 	45    	5.10096 	5.10096 	5.10096 	0      
38 	45    	-19.3471	-19.3471	-19.3471	0      
gen	nevals	avg     	min     	max     	std    
0  	50    	-404.356	-657.329	-122.977	155.241
37 	35    	115.724   	115.724 	115.724 	1.42109e-14
36 	41    	97.2136 	97.2136 	97.2136 	1.42109e-14
1  	38    	-262.761	-552.667	-59.8363	145.849
39 	40    	4.17825 	-10.2775	5.10096 	3.65219
39 	39    	-19.3471	-19.3471	-19.3471	0      
2  	39    	-167.015	-450.784	-59.8363	86.0999
38 	39    	115.724   	115.724 	115.724 	1.42109e-14
40 	39    	5.10096 	5.10096 	5.10096 	0      
Testing 105
37 	40    	97.2136 	97.2136 	97.2136 	1.42109e-14
40 	41    	-19.3471	-19.3471	-19.3471	0      
Testing 104
3  	38    	-134.495	-454.472	-59.8363	77.6003
39 	41    	115.724   	115.724 	115.724 	1.42109e-14
4  	36    	-88.3587	-185.53 	-9.85704	39.389 
38 	43    	97.2136 	97.2136 	97.2136 	1.42109e-14
Finished seed 10

...

8  	36    	-41.0229	-110.116	58.3794 	62.5245
25 	40    	59.9273 	59.9273 	59.9273 	7.10543e-15
40 	41    	93.8579 	85.7153 	94.0241 	1.16323
gen	nevals	avg    	min    	max     	std    
0  	50    	-429.31	-785.28	-94.6092	212.223
Testing 108
9  	39    	-11.8373	-77.8768	58.3794 	67.5063
37 	43    	1.01336 	1.01336 	1.01336 	0      
1  	41    	-221.914	-491.864	-71.4084	126.88 
10 	41    	18.7756 	-77.8768	58.3794 	60.5627
Finished seed 108 of algorithm lambda


/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


2  	42    	-150.614	-373.664	-60.0095	74.5719
26 	43    	59.9273 	59.9273 	59.9273 	7.10543e-15
11 	38    	43.0596 	-77.8768	58.3794 	41.7271
38 	46    	1.01336 	1.01336 	1.01336 	0      
3  	38    	-115.649	-491.864	-60.0095	69.0978
gen	nevals	avg     	min     	max     	std    
0  	50    	-421.765	-743.825	-2.95192	213.595
12 	35    	58.3794 	58.3794 	58.3794 	7.10543e-15
27 	43    	59.9273 	59.9273 	59.9273 	7.10543e-15
4  	45    	-99.1338	-305.678	-60.0095	47.9389
39 	38    	1.01336 	1.01336 	1.01336 	0      
1  	39    	-277.861	-743.825	-54.0524	175.433
5  	43    	-75.0838	-122.085	-60.0095	21.488 
13 	40    	58.3794 	58.3794 	58.3794 	7.10543e-15
2  	35    	-154.666	-484.902	-28.9893	92.5488
40 	43    	1.01336 	1.01336 	1.01336 	0      
Testing 109
28 	42    	59.9273 	59.9273 	59.9273 	7.10543e-15
6  	41    	-65.7725	-116.575	-60.0095	11.6278
3  	43    	-124.125	-195.156	-28.9893	37.7312
14 	40    	58.3794 	58.3794 	58.3794 	7.10543e-15
7  	41    	-59.3517	-78.6651	14.33   	11.056

/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


5  	37    	-108.808	-351.203	36.6521 	52.4367
8  	44    	-57.0359	-60.0095	14.33   	14.5675
15 	43    	58.3794 	58.3794 	58.3794 	7.10543e-15
30 	36    	59.9273 	59.9273 	59.9273 	7.10543e-15
gen	nevals	avg     	min     	max     	std    
0  	50    	-416.094	-715.152	-122.442	175.617
6  	43    	-74.39  	-150.827	36.6521 	50.8444
9  	37    	-57.0359	-60.0095	14.33   	14.5675
31 	43    	59.9273 	59.9273 	59.9273 	7.10543e-15
1  	44    	-254.82 	-574.92 	-103.671	138.006
7  	39    	-58.4507	-228.083	36.6521 	57.7379
16 	41    	58.3794 	58.3794 	58.3794 	7.10543e-15
10 	41    	-54.7421	-60.0095	14.33   	18.3413
2  	39    	-155.262	-482.592	-103.671	73.1616
8  	44    	-25.4638	-125.903	60.1728 	51.6929
32 	42    	59.9273 	59.9273 	59.9273 	7.10543e-15
11 	39    	-53.128 	-60.0095	14.33   	19.5429
17 	37    	58.3794 	58.3794 	58.3794 	7.10543e-15
3  	39    	-122.675	-217.046	-103.671	28.2666
9  	42    	-3.55673	-173.273	60.1728 	48.4708
12 	42    	-46.1634	-60.0095	21.5098 	26.7354
33 	35    